# Model A Final — Robustness & Deployment Validation (v2)
### Jianhui — Tree-Based Track (XGBoost + SMOTE)

**What changed from the previous version, and why it matters.** The
previous robustness notebook reimplemented preprocessing in its own
`build_model_ready_features` function and called the raw model directly.
That tested whether the *reimplementation* was robust — not whether the
actual deployed code (`inference.py`) is. This version calls
`inference.model_fn` / `inference.predict_fn` directly throughout, so every
result below is evidence about the real deployment artifact, not a parallel
copy of it that can silently drift out of sync.

**Model is loaded by registry alias, not a hard-coded version number.**
`models:/ITI113-team04-ModelA-XGBoost-Final@champion-candidate` always
resolves to whichever version is currently promoted — including a future
version promoted by the retraining pipeline — so this notebook does not
need to be manually updated when the champion changes.

**Four categories tested, per the module's AI Verify Alternative guidance:**
1. Noise — small perturbations to numeric features
2. Important-feature sweeps — varying top-ranked features across a realistic range
3. Missing values — each required field nulled independently
4. Out-of-range / edge cases — implausible or extreme values

Retained from the previous version: the deployment-contract consistency
check (Section 4), which caught the `amt_log` gap in an earlier run.


## 1. Environment Setup and MLflow Initialization

In [1]:
# ============================================================
# Install required packages
# ============================================================
%pip install -q -U \
    "boto3" \
    "botocore" \
    "mlflow==3.15.1" \
    "mlflow-skinny==3.15.1" \
    "mlflow-tracing==3.15.1" \
    "sagemaker-mlflow==0.5.0" \
    "xgboost" \
    "pyarrow"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import json
import warnings
from pathlib import Path
from urllib.parse import urlparse

import boto3
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

import mlflow
import mlflow.xgboost
from mlflow import MlflowClient
from mlflow_utils import initialize_mlflow

STUDENT_ID = "S402"
EXPERIMENT_NAME = "ITI113/team04/ModelA"
MLFLOW_APP_ARN = initialize_mlflow(student_id=STUDENT_ID, experiment_name=EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
assert mlflow.get_tracking_uri().startswith("arn:") or "sagemaker" in mlflow.get_tracking_uri().lower(), (
    "Tracking URI does not look like the SageMaker MLflow App -- "
    "initialize_mlflow() may not have run correctly. Do not proceed until this is fixed, "
    "or model loading below will silently fail against a local file store instead."
)


Initializing SageMaker MLflow connection for S402...
Target Experiment: ITI113/team04/ModelA
MLflow App ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-ANFQ3RACFV2G
MLflow Tracking URI successfully set.
Fresh MLflow UI URL:
https://app-ANFQ3RACFV2G.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IkFTREc3TyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHFTc2FTRzRlMmVtWDVRaHhvUWZEVVdrWnZ6U3NMTUVBL1hoR2xGVzhEd1VBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGeGRHUmFTMmRaVVcxb1dsVnhiWEpQUW5KM05FMU1RVXRIWW5sRE1XaDJWMnhMWVcxaGJsZFlVbGhrWlhaUFZXODJjRVZCWmtaNVprOVVRa3hzUlRCRVFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFXd3ZZeE5vYm1qdXQ1TmhnV1ppc0dZQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4cDhZa

In [3]:
# ---- inference.py must be on the path -- this notebook tests it directly ----
sys.path.insert(0, ".")  # adjust if inference.py lives in a different folder
import inference

print("inference.py loaded from:", inference.__file__)


inference.py loaded from: /home/sagemaker-user/Jianhui/inference.py


## 2. Load the Real Registered Model, Contract, and Preprocessing Bundle

Everything is loaded by alias from the registry and S3 — nothing here is
hard-coded to a specific version or a locally-cached file that could go
stale.

In [4]:
S3_BUCKET_URI = "s3://nyp-26s1-iti113/iti113/team04/data/credit-card-fraud-detection/"
S3_MODELA_OUTPUT_URI = S3_BUCKET_URI + "processed/modela_baseline/"
LOCAL_MODEL_DIR = Path("./model_dir_robustness_test")
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_REGISTERED_MODEL_NAME = "ITI113-team04-ModelA-XGBoost-Final"
CHAMPION_ALIAS = "champion-candidate"

DEPLOYMENT_CONTRACT_FILENAME = "model_a_deployment_contract.json"
PREPROCESSING_BUNDLE_FILENAME = "model_a_preprocessing_bundle.joblib"

client = MlflowClient()
champion_version_info = client.get_model_version_by_alias(FINAL_REGISTERED_MODEL_NAME, CHAMPION_ALIAS)
print(f"Registry: '{CHAMPION_ALIAS}' -> {FINAL_REGISTERED_MODEL_NAME} version {champion_version_info.version}")
print(f"Source run: {champion_version_info.run_id}")

# Load the real model straight from the registry (by alias) and save it
# locally in the exact shape inference.model_fn() expects.
champion_model = mlflow.xgboost.load_model(f"models:/{FINAL_REGISTERED_MODEL_NAME}@{CHAMPION_ALIAS}")
champion_model.save_model(str(LOCAL_MODEL_DIR / "xgboost_model.json"))

def _download(s3_key, local_path):
    bucket = "nyp-26s1-iti113"
    key = "iti113/team04/data/credit-card-fraud-detection/processed/modela_baseline/" + s3_key
    boto3.client("s3").download_file(bucket, key, str(local_path))

_download(DEPLOYMENT_CONTRACT_FILENAME, LOCAL_MODEL_DIR / DEPLOYMENT_CONTRACT_FILENAME)
_download(PREPROCESSING_BUNDLE_FILENAME, LOCAL_MODEL_DIR / PREPROCESSING_BUNDLE_FILENAME)

model_bundle = inference.model_fn(str(LOCAL_MODEL_DIR))
FEATURE_COLUMNS = model_bundle["contract"]["feature_columns"]
RAW_FIELDS_REQUIRED = model_bundle["contract"]["raw_input_fields_required"]
OPERATING_THRESHOLD = model_bundle["operating_threshold"]

print(f"\nLoaded via inference.model_fn(). Operating threshold: {OPERATING_THRESHOLD}")
print(f"Model feature columns: {FEATURE_COLUMNS}")
print(f"Raw fields required per contract: {RAW_FIELDS_REQUIRED}")


Registry: 'champion-candidate' -> ITI113-team04-ModelA-XGBoost-Final version 5
Source run: 637cb783c2be4269a50132c206cd36a5



Loaded via inference.model_fn(). Operating threshold: 0.94
Model feature columns: ['amt', 'amt_log', 'category_te', 'distance_km', 'distance_log', 'city_pop', 'trans_hour', 'day_of_week', 'is_weekend', 'age', 'gender_binary']
Raw fields required per contract: ['amt', 'category', 'distance_km', 'distance_log', 'city_pop', 'trans_hour', 'day_of_week', 'is_weekend', 'age', 'gender']


## 3. Smoke Test — Confirm the Loaded Model Matches Its Recorded Metrics

Before trusting anything below, confirm the model loaded here reproduces
what the registry says it should, the same reconciliation discipline used
throughout this project's other notebooks.

In [5]:
results_s3_key = "model_a_final_results.json"
try:
    _download(results_s3_key, LOCAL_MODEL_DIR / results_s3_key)
    with open(LOCAL_MODEL_DIR / results_s3_key) as f:
        recorded_results = json.load(f)
    print("Recorded final-model results loaded for comparison.")
    print(json.dumps(recorded_results.get("test_metrics", recorded_results), indent=2)[:600])
except Exception as exc:
    print(f"⚠ Could not load recorded results for reconciliation ({type(exc).__name__}: {exc}).")
    print("Proceeding without a smoke-test comparison -- results below are not yet reconciled against a recorded baseline.")


Recorded final-model results loaded for comparison.
{
  "schema_version": "2.0",
  "model_stage": "final",
  "student_id": "S402",
  "track": "Tree-Based Pipeline (Model A)",
  "candidate": "full_11_features_baseline_params",
  "feature_count": 11,
  "feature_columns": [
    "amt",
    "amt_log",
    "category_te",
    "distance_km",
    "distance_log",
    "city_pop",
    "trans_hour",
    "day_of_week",
    "is_weekend",
    "age",
    "gender_binary"
  ],
  "selected_threshold": 0.94,
  "selection_basis": "validation_only",
  "threshold_grid": "0.05_to_1.00_step_0.01",
  "dataset": {
    "train_rows": 778005,
    "validation_rows": 259335,
 


## 4. Deployment Contract Consistency Check

Retained from the previous version — this is the automated check that
catches exactly the class of gap found in an earlier manual run (a model
feature not covered by the documented raw input contract).

In [6]:
def _apply_target_encoding(df, model_bundle):
    """Feature-Engineering-level fields -> model-ready numeric matrix."""
    df = df.copy()
    bundle = model_bundle["preprocessing_bundle"]
    contract = model_bundle["contract"]

    te_mapping = bundle["category_te_mapping"]
    te_global_mean = bundle["category_te_global_mean"]
    gender_mapping = bundle["gender_mapping"]

    df["category_te"] = df["category"].map(te_mapping).fillna(te_global_mean).astype(float)
    df["gender_binary"] = df["gender"].map(gender_mapping)

    # amt_log is a deterministic transform of amt, not something a caller
    # should need to compute and supply separately -- derive it here,
    # same treatment as category_te and gender_binary above.
    if "amt_log" not in df.columns and "amt" in df.columns:
        df["amt_log"] = np.log1p(df["amt"].clip(lower=0))

    feature_columns = contract["feature_columns"]
    missing = [c for c in feature_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Preprocessing did not produce all required model features: {missing}")

    return df[feature_columns].apply(pd.to_numeric, errors="coerce")

In [7]:
documented_raw = set(RAW_FIELDS_REQUIRED)
documented_features = set(FEATURE_COLUMNS)
derived_by_target_encoding = {"category_te", "gender_binary", "amt_log"}

# Every model feature must be either a documented raw input, or something
# inference.py's own preprocessing derives (category_te, gender_binary).
# Anything else is a gap: the model needs it, but nothing produces it.
unaccounted = documented_features - documented_raw - derived_by_target_encoding

print("Model features:", sorted(documented_features))
print("Documented raw inputs:", sorted(documented_raw))
print("Derived by preprocessing (category_te, gender_binary):", sorted(derived_by_target_encoding))
print()
if unaccounted:
    print(f"⚠ CONTRACT GAP: these model features are neither documented as required raw "
          f"input, nor produced by preprocessing: {sorted(unaccounted)}")
    print("A caller providing exactly the documented required fields will fail on these.")
else:
    print("✅ Contract is internally consistent -- every model feature is accounted for.")


Model features: ['age', 'amt', 'amt_log', 'category_te', 'city_pop', 'day_of_week', 'distance_km', 'distance_log', 'gender_binary', 'is_weekend', 'trans_hour']
Documented raw inputs: ['age', 'amt', 'category', 'city_pop', 'day_of_week', 'distance_km', 'distance_log', 'gender', 'is_weekend', 'trans_hour']
Derived by preprocessing (category_te, gender_binary): ['amt_log', 'category_te', 'gender_binary']

✅ Contract is internally consistent -- every model feature is accounted for.


## 5. Reference Transactions

Two illustrative transactions, matching Feature-Engineering-level output
(post-transform, pre-target-encoding), used as the baseline for every
perturbation test below. Because the contract gap in Section 4 may still be
present, `amt_log` is included explicitly here so the remaining tests are
not blocked by it — the gap itself is already captured as its own finding
above, not silently hidden by this workaround.

In [8]:
reference_transactions = pd.DataFrame([
    {  # low amount, daytime
        "amt": 18.50, "amt_log": np.log1p(18.50), "category": "grocery_pos",
        "distance_km": 5.4, "distance_log": np.log1p(5.4),
        "city_pop": 62000, "trans_hour": 13, "day_of_week": 2, "is_weekend": 0,
        "age": 39, "gender": "F",
    },
    {  # high amount, late night
        "amt": 940.00, "amt_log": np.log1p(940.00), "category": "shopping_net",
        "distance_km": 132.0, "distance_log": np.log1p(132.0),
        "city_pop": 9800, "trans_hour": 23, "day_of_week": 5, "is_weekend": 1,
        "age": 29, "gender": "M",
    },
])

reference_result = inference.predict_fn(reference_transactions, model_bundle)
display(pd.concat([reference_transactions[["amt", "category", "trans_hour"]], reference_result], axis=1))


,amt,category,trans_hour,fraud_probability,is_fraud_predicted,operating_threshold
0,18.5,grocery_pos,13,0.000542,0,0.94
1,940.0,shopping_net,23,0.978370,1,0.94


## 6. Noise — Small Perturbations to Numeric Features

Each numeric feature perturbed with Gaussian noise scaled to a fraction of
its own magnitude, across many trials, checking the real model's
probability output for stability.

In [9]:
NOISE_LEVELS = [0.01, 0.05, 0.10]
N_TRIALS = 200
RNG = np.random.default_rng(42)
NUMERIC_FEATURES = ["amt", "distance_km", "city_pop", "age"]

noise_rows = []
for ref_idx, base_row in reference_transactions.iterrows():
    base_df = pd.DataFrame([base_row])
    base_pred = inference.predict_fn(base_df, model_bundle)
    base_prob = base_pred["fraud_probability"].iloc[0]
    base_class = base_pred["is_fraud_predicted"].iloc[0]

    for feature in NUMERIC_FEATURES:
        for level in NOISE_LEVELS:
            probs, flips = [], 0
            for _ in range(N_TRIALS):
                noisy_df = base_df.copy()
                noise = RNG.normal(0, level * max(abs(base_row[feature]), 1.0))
                noisy_df[feature] = base_row[feature] + noise
                if feature == "distance_km":
                    noisy_df["distance_log"] = np.log1p(noisy_df["distance_km"].clip(lower=0))
                if feature == "amt":
                    noisy_df["amt_log"] = np.log1p(noisy_df["amt"].clip(lower=0))
                noisy_pred = inference.predict_fn(noisy_df, model_bundle)
                probs.append(noisy_pred["fraud_probability"].iloc[0])
                if noisy_pred["is_fraud_predicted"].iloc[0] != base_class:
                    flips += 1
            noise_rows.append({
                "reference_amt": base_row["amt"], "feature": feature, "noise_level": level,
                "base_probability": round(base_prob, 4),
                "mean_probability_shift": round(float(np.mean(np.abs(np.array(probs) - base_prob))), 4),
                "max_probability_shift": round(float(np.max(np.abs(np.array(probs) - base_prob))), 4),
                "class_flips_out_of_trials": f"{flips}/{N_TRIALS}",
            })

display(pd.DataFrame(noise_rows))


,reference_amt,feature,noise_level,base_probability,mean_probability_shift,max_probability_shift,class_flips_out_of_trials
0,18.5,amt,0.01,0.0005,0.0000,0.0000,0/200
1,18.5,amt,0.05,0.0005,0.0000,0.0001,0/200
2,18.5,amt,0.10,0.0005,0.0000,0.0002,0/200
3,18.5,distance_km,0.01,0.0005,0.0000,0.0000,0/200
4,18.5,distance_km,0.05,0.0005,0.0000,0.0000,0/200
5,18.5,distance_km,0.10,0.0005,0.0000,0.0000,0/200
6,18.5,city_pop,0.01,0.0005,0.0000,0.0000,0/200
7,18.5,city_pop,0.05,0.0005,0.0000,0.0000,0/200
8,18.5,city_pop,0.10,0.0005,0.0000,0.0000,0/200
9,18.5,age,0.01,0.0005,0.0000,0.0000,0/200


## 7. Important-Feature Sweeps

In [10]:
def sweep_feature(base_row, feature, values):
    rows = []
    for v in values:
        df = pd.DataFrame([base_row])
        df[feature] = v
        if feature == "amt":
            df["amt_log"] = np.log1p(max(v, 0))
        pred = inference.predict_fn(df, model_bundle)
        rows.append({feature: v, "fraud_probability": pred["fraud_probability"].iloc[0]})
    return pd.DataFrame(rows)

base = reference_transactions.iloc[0]

amt_sweep = sweep_feature(base, "amt", [5, 25, 50, 100, 250, 500, 1000, 2000, 5000])
print("Amount sweep (compare against the $50-$250 blind spot found in the fairness audit):")
display(amt_sweep)

age_sweep = sweep_feature(base, "age", [18, 25, 35, 45, 55, 65, 75, 90])
print("Age sweep:")
display(age_sweep)


Amount sweep (compare against the $50-$250 blind spot found in the fairness audit):


,amt,fraud_probability
0,5,0.000048
1,25,0.000344
2,50,0.000216
3,100,0.000324
4,250,0.009746
5,500,0.925061
6,1000,0.965101
7,2000,0.837648
8,5000,0.837648


Age sweep:


,age,fraud_probability
0,18,0.000332
1,25,0.000438
2,35,0.000438
3,45,0.000566
4,55,0.001662
5,65,0.001442
6,75,0.001473
7,90,0.001047


## 8. Missing Values — Each Required Field Nulled Independently

Every field the deployment contract lists as required is set to null, one
at a time, calling `inference.predict_fn` directly — this tests the real
code, so the result reflects what a production caller would actually
experience, not a reimplementation's behaviour.

In [11]:
missing_value_results = []
base = reference_transactions.iloc[0]

for field in RAW_FIELDS_REQUIRED:
    test_df = pd.DataFrame([base]).copy()
    test_df[field] = None
    try:
        pred = inference.predict_fn(test_df, model_bundle)
        prob = pred["fraud_probability"].iloc[0]
        outcome = "SILENT NaN" if pd.isna(prob) else "produced a prediction (no error)"
        missing_value_results.append({"field": field, "outcome": outcome, "fraud_probability": prob, "note": ""})
    except Exception as exc:
        missing_value_results.append({"field": field, "outcome": "REJECTED (raised an error)",
                                       "fraud_probability": None, "note": f"{type(exc).__name__}: {exc}"})

missing_df = pd.DataFrame(missing_value_results)
display(missing_df)


,field,outcome,fraud_probability,note
0,amt,produced a prediction (no error),0.808849,
1,category,produced a prediction (no error),0.739818,
2,distance_km,produced a prediction (no error),0.028746,
3,distance_log,produced a prediction (no error),0.000539,
4,city_pop,produced a prediction (no error),0.001221,
5,trans_hour,produced a prediction (no error),0.001423,
6,day_of_week,produced a prediction (no error),0.001282,
7,is_weekend,produced a prediction (no error),0.000336,
8,age,produced a prediction (no error),0.000903,
9,gender,produced a prediction (no error),0.000356,


**How to read this table.** `"produced a prediction (no error)"` for a
field that was set to null is the finding to scrutinise — it means
`inference.py` currently has no explicit validation for that field, and
whatever happened internally (an implicit default, or XGBoost's native
missing-value handling) happened silently. `"REJECTED"` means the code
explicitly caught the problem. Neither outcome is inherently wrong, but only
one of them is a *deliberate* design decision — worth checking against
`inference.py`'s actual source for each row here.

## 9. Out-of-Range and Edge-Case Values

In [12]:
edge_cases_specs = [
    ("amt=0", {"amt": 0.0, "amt_log": np.log1p(0.0)}),
    ("amt<0", {"amt": -50.0, "amt_log": np.log1p(0.0)}),
    ("amt=999999", {"amt": 999999.0, "amt_log": np.log1p(999999.0)}),
    ("age=0", {"age": 0}),
    ("age=150", {"age": 150}),
    ("unseen category", {"category": "not_a_real_category"}),
    ("city_pop<0", {"city_pop": -1}),
    ("unrecognised gender", {"gender": "X"}),
]

edge_results = []
base = reference_transactions.iloc[0]
for label, overrides in edge_cases_specs:
    row = pd.DataFrame([base]).copy()
    for k, v in overrides.items():
        row[k] = v
    try:
        pred = inference.predict_fn(row, model_bundle)
        edge_results.append({"case": label, "outcome": "handled (no error)",
                              "fraud_probability": pred["fraud_probability"].iloc[0]})
    except Exception as exc:
        edge_results.append({"case": label, "outcome": "REJECTED (raised an error)",
                              "fraud_probability": None, "note": f"{type(exc).__name__}: {exc}"})

display(pd.DataFrame(edge_results))


,case,outcome,fraud_probability
0,amt=0,handled (no error),0.000030
1,amt<0,handled (no error),0.000030
2,amt=999999,handled (no error),0.837648
3,age=0,handled (no error),0.000332
4,age=150,handled (no error),0.000903
5,unseen category,handled (no error),0.739818
6,city_pop<0,handled (no error),0.001025
7,unrecognised gender,handled (no error),0.000356


## 10. Summary — for Direct Inclusion in the Test Report

Every result above comes from calling `inference.model_fn` / `predict_fn`
against the real registered model (version confirmed in Section 2), so
these findings describe the actual deployment artifact.

In [13]:
print("ROBUSTNESS TEST SUMMARY -- tested against inference.py + the real registered model")
print("=" * 70)
print(f"Model: {FINAL_REGISTERED_MODEL_NAME} @ {CHAMPION_ALIAS} (version {champion_version_info.version})")
print()
print("1. Deployment contract consistency: see Section 4.")
print("2. Noise stability: see Section 6.")
print("3. Feature sweeps: see Section 7 -- cross-reference the amount sweep against")
print("   the fairness audit's $50-$250 blind spot finding if a similar dip appears.")
print("4. Missing-value handling, per field:")
display(missing_df[["field", "outcome"]])
print()
print("5. Edge cases:")
display(pd.DataFrame(edge_results)[["case", "outcome"]])


ROBUSTNESS TEST SUMMARY -- tested against inference.py + the real registered model
Model: ITI113-team04-ModelA-XGBoost-Final @ champion-candidate (version 5)

1. Deployment contract consistency: see Section 4.
2. Noise stability: see Section 6.
3. Feature sweeps: see Section 7 -- cross-reference the amount sweep against
   the fairness audit's $50-$250 blind spot finding if a similar dip appears.
4. Missing-value handling, per field:


,field,outcome
0,amt,produced a prediction (no error)
1,category,produced a prediction (no error)
2,distance_km,produced a prediction (no error)
3,distance_log,produced a prediction (no error)
4,city_pop,produced a prediction (no error)
5,trans_hour,produced a prediction (no error)
6,day_of_week,produced a prediction (no error)
7,is_weekend,produced a prediction (no error)
8,age,produced a prediction (no error)
9,gender,produced a prediction (no error)



5. Edge cases:


,case,outcome
0,amt=0,handled (no error)
1,amt<0,handled (no error)
2,amt=999999,handled (no error)
3,age=0,handled (no error)
4,age=150,handled (no error)
5,unseen category,handled (no error)
6,city_pop<0,handled (no error)
7,unrecognised gender,handled (no error)
